# The Tool-Using Agent — Build It Safe

**What you'll learn:** how an LLM agent loop works, why safety guardrails are necessary, and how to build a tool registry with validation.

**Estimated time:** 60–90 min &nbsp;|&nbsp; **Runtime:** CPU. You'll need an OpenAI API key.

---

### Before any code: think about the threat model

An agent loop is simple in principle:
1. Give the LLM a goal
2. It decides what tool to call
3. You execute the tool
4. Feed the result back to the LLM
5. Repeat until it says it's done

? **What could go wrong?** Brainstorm at least 3 things that could go badly if you naïvely execute whatever the LLM tells you. Write them down before reading further.

Now consider: what constraints would prevent each of your scenarios?


## 1  Setup and API key

In [ ]:
!pip -q install openai
import os, json, re
from openai import OpenAI

# Paste your API key below — do NOT commit this notebook to a public repo
os.environ['OPENAI_API_KEY'] = 'sk-...'
client = OpenAI()

# Quick test to confirm the key works
try:
    r = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[{'role': 'user', 'content': 'Reply only with: OK'}],
        max_tokens=5,
    )
    print("API connection:", r.choices[0].message.content.strip())
except Exception as e:
    print("API error:", e)

## 2  Define safe tools — and think about what you leave out

Each tool is a plain Python function. The **tool registry** (`TOOLS`) is the only way the agent can take action in the world. Anything not in the registry is impossible.

Notice the deliberate restrictions:
- No `eval` of arbitrary Python
- No filesystem write access
- No network calls
- No shell execution

? **Why these restrictions?** Consider: if you gave the agent a tool that could `eval` arbitrary Python, what could it do?


In [ ]:
import math

def calculator(expr: str) -> str:
    # Evaluate a math expression safely - only allows basic arithmetic
    # eval() is used here intentionally and is safe because:
    #   1. The regex below is an allow-list that rejects any character outside
    #      digits, +−*/().  and whitespace — no letters, no builtins, no imports.
    #   2. eval() is called with {"__builtins__": {}} and {} so even if the
    #      regex were bypassed, no built-in names or variables are accessible.
    # This is the standard two-layer defense for a math-only sandbox.
    if not re.match(r'^[0-9+\-*/().\s]+$', expr):
        return "rejected: only arithmetic expressions allowed"
    try:
        result = eval(expr, {"__builtins__": {}}, {})
        return str(round(float(result), 6))
    except Exception as e:
        return f"error: {e}"

def word_count(text: str) -> str:
    # Count words in a string
    return str(len(text.split()))

def to_uppercase(text: str) -> str:
    # Convert text to uppercase
    return text.upper()

# YOUR TASK: add one more safe tool below
# Ideas: count_vowels, reverse_string, count_sentences, estimate_reading_time
# def your_tool(text: str) -> str:
#     ...

def count_sentences(text: str) -> str:
    # Count sentences by sentence-ending punctuation
    return str(len(re.findall(r'[.!?]+', text)))

TOOLS = {
    'calculator':    calculator,
    'word_count':    word_count,
    'to_uppercase':  to_uppercase,
    'count_sentences': count_sentences,
}

print("Registered tools:", list(TOOLS.keys()))
print()
# Test your tools directly before trusting the agent with them
print("calculator('2 ** 10'):", calculator('2 ** 10'))
print("word_count('hello world foo'):", word_count('hello world foo'))
print("calculator('__import__("os")'):", calculator('__import__("os")'))

? **Notice the last test.** The `calculator` function rejected the import attempt. Why? What part of the code did that?


## 3  Build the validation layer

Before any tool call executes, it must pass through a validation layer. This is the security boundary — the place where we decide whether to trust the agent's request.

Fill in the `run_tool` function:


In [ ]:
def run_tool(name: str, arg: str) -> str:
    # Validate and execute a tool call from the agent

    # Validation 1: is this tool in our allow-list?
    # YOUR CODE HERE — return an error string if name not in TOOLS
    if name not in TOOLS:
        return f"rejected: '{name}' is not a registered tool. Available: {list(TOOLS.keys())}"

    # Validation 2: sanity-check the argument length
    if len(str(arg)) > 1000:
        return "rejected: argument too long (max 1000 chars)"

    # Execute and catch any errors
    try:
        result = TOOLS[name](str(arg))
        return str(result)[:500]   # cap output length too
    except Exception as e:
        return f"tool error: {e}"

# Test validation
print("Known tool:   ", run_tool('calculator', '3 * 7'))
print("Unknown tool: ", run_tool('delete_files', '/etc/passwd'))
print("Oversized arg:", run_tool('word_count', 'x' * 2000))

## 4  The agent loop with a step cap

The system prompt tells the model the tool API. It must reply with JSON in one of two forms:
- `{"tool": "name", "arg": "value"}` — wants to call a tool
- `{"final": "answer"}` — done

The step cap prevents infinite loops. Without it, a buggy or misbehaving model could run forever.

? **Before running:** Without the `max_steps` cap, what would happen if the model kept calling tools in a loop? What real-world cost would that have?


In [ ]:
SYSTEM_PROMPT = (
    "You are an assistant with access to these tools: " + str(list(TOOLS.keys())) + ".\n\n"
    "To use a tool, reply ONLY with valid JSON:\n"
    "  {'tool': 'tool_name', 'arg': 'argument_string'}\n\n"
    "When you have the final answer and need no more tools, reply ONLY with:\n"
    "  {'final': 'your complete answer'}\n\n"
    "Never add any text outside the JSON. Never call a tool not in the list."
)

def agent(goal: str, max_steps: int = 6) -> str:
    history = [
        {'role': 'system',  'content': SYSTEM_PROMPT},
        {'role': 'user',    'content': goal},
    ]
    print(f"Goal: {goal}")
    print("-" * 50)

    for step in range(max_steps):
        response = client.chat.completions.create(
            model='gpt-4o-mini',
            messages=history,
            max_tokens=200,
            temperature=0,   # deterministic for debugging
        )
        raw = response.choices[0].message.content.strip()
        history.append({'role': 'assistant', 'content': raw})

        # Parse the model's JSON reply
        try:
            parsed = json.loads(raw)
        except json.JSONDecodeError:
            print(f"  Step {step+1}: model returned non-JSON — stopping")
            print(f"  Raw output: {repr(raw)}")
            return "Agent error: model did not return valid JSON"

        if 'final' in parsed:
            print(f"  Step {step+1}: DONE -> {parsed['final']}")
            return parsed['final']

        elif 'tool' in parsed:
            tool_name = parsed.get('tool', '')
            tool_arg  = parsed.get('arg', '')
            result = run_tool(tool_name, tool_arg)
            print(f"  Step {step+1}: {tool_name}({repr(tool_arg)}) -> {repr(result)}")
            history.append({'role': 'user', 'content': f"Tool result: {result}"})

        else:
            return "Agent error: unexpected JSON shape"

    return f"Agent stopped after {max_steps} steps without a final answer"

print("Agent loop ready.")

## 5  Run the agent on several tasks

? **Predict** which tasks will require 1 step, which will require 2+, and which might fail.


In [ ]:
# Simple single-tool task
result = agent("What is 144 divided by 12, then squared?")
print("\nFinal answer:", result)

In [ ]:
print()
# Multi-tool task
result = agent("How many words are in the phrase 'To be or not to be that is the question', and what is that number times 3?")
print("\nFinal answer:", result)

In [ ]:
print()
# Task that uses your custom tool
result = agent("Convert 'machine learning is fascinating' to uppercase and count its words.")
print("\nFinal answer:", result)

? **Reflect:**
- Did any task take more steps than you predicted?
- Did the agent ever make a mistake? How did the validation layer respond?
- What would happen if you gave the agent a goal that requires a tool it doesn't have?


## 6  Add human approval for risky actions

Some tools should require human sign-off before executing — for example, anything that writes to disk or sends a message.

Your task: modify `run_tool` to add a `REQUIRES_APPROVAL` set, and prompt the user (via `input()`) when a risky tool is called.


In [ ]:
REQUIRES_APPROVAL = {'delete_file', 'send_email', 'write_file'}

def run_tool_with_approval(name: str, arg: str) -> str:
    # Like run_tool, but pauses for human approval on risky tools
    if name not in TOOLS:
        return f"rejected: '{name}' is not registered"

    # YOUR CODE HERE — check if tool requires approval, prompt user, proceed or abort
    if name in REQUIRES_APPROVAL:
        print(f"\nWARNING️  APPROVAL REQUIRED: {name}({repr(arg[:80])})")
        answer = input("Allow? (y/n): ").strip().lower()
        if answer != 'y':
            return "rejected: user denied the action"

    try:
        return str(TOOLS[name](str(arg)))[:500]
    except Exception as e:
        return f"tool error: {e}"

# Simulate a risky tool call (uses input(), so this cell is interactive)
# print(run_tool_with_approval('to_uppercase', 'test'))       # safe — no approval needed
# print(run_tool_with_approval('send_email', 'foo@bar.com'))  # risky — would prompt

## 7  Final challenges

**Challenge 1 — Rate limiting:** Add a counter that tracks how many tool calls have been made in the last 60 seconds. Reject calls that would exceed 10/min.

**Challenge 2 — Audit log:** Write every tool call and result to a log file (`audit.jsonl`, one JSON object per line). A real production agent should always have an audit trail.

**Challenge 3 — Agent that uses itself recursively:** What would happen if you gave the agent a `sub_agent` tool that called `agent()` again? Is this safe? Why or why not?

**Challenge 4 — Adversarial prompt:** Try to get the agent to call a tool not in its list by crafting a clever goal (prompt injection). What happens? Can you make the validation layer more robust?

These are the exact challenges that production AI teams think about every day.
